# Web Scraping with Python
Using `requests` and `BeautifulSoup` to scrape, parse, and save web data.

In [ ]:
# Import Required Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import csv
import time

In [ ]:
# Send HTTP Request to a Web Page
url = "http://books.toscrape.com/"

response = requests.get(url, timeout=5)
print(f"Status Code: {response.status_code}")
print(f"Content Length: {len(response.text)} characters")

In [ ]:
# Parse HTML Content with BeautifulSoup
soup = BeautifulSoup(response.content, "html.parser")

# Preview the parsed HTML structure
print(soup.prettify()[:500])

In [ ]:
# Extract Specific Elements by Tag and Class
# Get all book titles and prices from the page
books = soup.find_all("article", class_="product_pod")

for book in books[:5]:  # Preview first 5
    title = book.find("h3").find("a")["title"]
    price = book.find("p", class_="price_color").text
    print(f"{title} — {price}")

In [ ]:
# Extract Links and Attributes
# Find all anchor tags and get their href attributes
links = soup.find_all("a")

for link in links[:10]:  # Preview first 10 links
    href = link.get("href", "No link")
    text = link.get_text(strip=True)
    print(f"{text} → {href}")

In [ ]:
# Extract All Books into a DataFrame
all_books = []

for book in books:
    title = book.find("h3").find("a")["title"]
    price = book.find("p", class_="price_color").text
    stock = book.find("p", class_="instock availability").get_text(strip=True)
    all_books.append({"Title": title, "Price": price, "Stock": stock})

df = pd.DataFrame(all_books)
df.head()

In [ ]:
# Handle Pagination — Scrape Multiple Pages
from urllib.parse import urljoin

base_url = "http://books.toscrape.com/catalogue/"
current_url = urljoin(base_url, "page-1.html")
all_books = []

while current_url:
    response = requests.get(current_url, timeout=5)
    soup = BeautifulSoup(response.content, "html.parser")
    
    books = soup.find_all("article", class_="product_pod")
    for book in books:
        title = book.find("h3").find("a")["title"]
        price = book.find("p", class_="price_color").text
        all_books.append({"Title": title, "Price": price})
    
    # Check for next page
    next_btn = soup.find("li", class_="next")
    if next_btn:
        next_link = next_btn.find("a")["href"]
        current_url = urljoin(current_url, next_link)
        time.sleep(1)  # Be polite to the server
    else:
        current_url = None

print(f"Scraped {len(all_books)} books total.")

In [ ]:
# Save Scraped Data to CSV
df = pd.DataFrame(all_books)
df.to_csv("scraped_books.csv", index=False)
print(f"Saved {len(df)} books to scraped_books.csv")
df.head(10)